# Week 9: Gradient Boosting
## Digital Customer Journey and E-Commerce Retention Analytics

**Student:** Javed Aamer

This analysis uses gradient boosting to predict `OrderCount` from customer-level e-commerce data.

`OrderCount` works well for this exercise because gradient-boosting regression can be followed from the initial prediction through successive corrections to the residuals. I use that structure to examine how the model changes with the **learning rate**, **number of estimators**, **tree depth**, and **regularization**.

The final model is selected from validation results and evaluated once on a separate test set.

## Focus for Week 9

The analysis covers:

- gradient boosting as a sequence of weak learners;
- residual fitting in regression;
- learning rate;
- number of estimators;
- the interaction between learning rate and the number of trees;
- tree depth and model complexity;
- underfitting and overfitting;
- minimum samples per leaf;
- stochastic boosting through subsampling;
- validation-based model selection;
- RMSE, MAE, and R-squared for evaluation.

Because the model is tree based, numeric features do not need to be standardized.

## 1. Questions guiding the analysis

1. What does the first residual-correction step look like for `OrderCount`?
2. How does the learning rate change the number of trees needed?
3. When does increasing tree depth begin to improve training fit more than validation performance?
4. Do subsampling and minimum leaf size improve generalization?
5. Which gradient-boosting configuration performs best on validation data?
6. Which customer characteristics contribute most to the final model?
7. How large is the gap between development and test performance?

In [1]:
# Imports and reproducibility
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

RANDOM_STATE = 42

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [2]:
# Load the cleaned customer-level dataset
candidate_paths = [
    Path("data/churn_clean.csv"),
    Path("churn_clean.csv"),
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find churn_clean.csv. Put it in a data/ folder next to this notebook."
    )

churn = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH}")
print(f"Shape: {churn.shape[0]:,} rows x {churn.shape[1]} columns")
print(f"Unique CustomerID values: {churn['CustomerID'].nunique():,}")
print(f"Duplicate CustomerID rows: {churn['CustomerID'].duplicated().sum():,}")

display(churn.head())

Loaded: data/churn_clean.csv
Shape: 5,630 rows x 20 columns
Unique CustomerID values: 5,630
Duplicate CustomerID rows: 0


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0000,Mobile Phone,3,6.0000,Debit Card,Female,3.0000,3,Laptop & Accessory,2,Single,9,1,11.0000,1.0000,1.0000,5.0000,159.9300
1,50002,1,9.0000,Phone,1,8.0000,UPI,Male,3.0000,4,Mobile,3,Single,7,1,15.0000,0.0000,1.0000,0.0000,120.9000
2,50003,1,9.0000,Phone,1,30.0000,Debit Card,Male,2.0000,4,Mobile,3,Single,6,1,14.0000,0.0000,1.0000,3.0000,120.2800
3,50004,1,0.0000,Phone,3,15.0000,Debit Card,Male,2.0000,4,Laptop & Accessory,5,Single,8,0,23.0000,0.0000,1.0000,3.0000,134.0700
4,50005,1,0.0000,Phone,1,12.0000,CC,Male,3.0000,3,Mobile,5,Single,3,0,11.0000,1.0000,1.0000,3.0000,129.6000


## 2. Modeling setup

I use the same customer-level predictors as in the recent retention analyses so the Week 9 results remain comparable with the earlier models.

### Response
- `OrderCount`

### Numeric predictors
`Tenure`, `CityTier`, `WarehouseToHome`, `HourSpendOnApp`, `NumberOfDeviceRegistered`, `SatisfactionScore`, `NumberOfAddress`, `Complain`, `OrderAmountHikeFromlastYear`, `CouponUsed`, `DaySinceLastOrder`, and `CashbackAmount`.

### Categorical predictors
`PreferredLoginDevice`, `PreferredPaymentMode`, and `PreferedOrderCat`.

`OrderCount` is removed from the predictor set because it is the response. `CustomerID` is excluded because it is an identifier. `Gender` and `MaritalStatus` are not included in this feature set.

Gradient-boosted trees split on feature thresholds, so standardization is not required.

In [3]:
reg_target = "OrderCount"

numeric_features = [
    "Tenure",
    "CityTier",
    "WarehouseToHome",
    "HourSpendOnApp",
    "NumberOfDeviceRegistered",
    "SatisfactionScore",
    "NumberOfAddress",
    "Complain",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
    "CashbackAmount",
]

categorical_features = [
    "PreferredLoginDevice",
    "PreferredPaymentMode",
    "PreferedOrderCat",
]

reg_numeric_features = [x for x in numeric_features if x != reg_target]
reg_features = reg_numeric_features + categorical_features

audit = pd.DataFrame({
    "dtype": churn[reg_features + [reg_target]].dtypes.astype(str),
    "missing": churn[reg_features + [reg_target]].isna().sum(),
    "unique": churn[reg_features + [reg_target]].nunique(dropna=False),
})
display(audit)

,dtype,missing,unique
Tenure,float64,0,36
CityTier,int64,0,3
WarehouseToHome,float64,0,34
HourSpendOnApp,float64,0,6
NumberOfDeviceRegistered,int64,0,6
SatisfactionScore,int64,0,5
NumberOfAddress,int64,0,15
Complain,int64,0,2
OrderAmountHikeFromlastYear,float64,0,16
CouponUsed,float64,0,17


## 3. Response distribution

`OrderCount` is right-skewed and discrete, with many customers placing only a small number of orders.

That shape makes RMSE useful for emphasizing larger prediction errors, while MAE provides a complementary measure that is less dominated by a few large misses. R-squared is also reported to show how much variation in ordering activity the model explains.

In [4]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(churn["OrderCount"], bins=np.arange(0.5, churn["OrderCount"].max() + 1.5, 1))
ax.set_xlabel("OrderCount")
ax.set_ylabel("Number of customers")
ax.set_title("Distribution of OrderCount")
ax.grid(axis="y")
plt.tight_layout()
plt.show()

print(f"Mean OrderCount: {churn['OrderCount'].mean():.2f}")
print(f"Median OrderCount: {churn['OrderCount'].median():.2f}")

<Figure size 800x500 with 1 Axes>

Mean OrderCount: 2.96
Median OrderCount: 2.00


## 4. Train, validation, and test split

I use a 60/20/20 split:

- the training set fits candidate models;
- the validation set is used to compare learning rate, estimator count, depth, and regularization;
- the test set is held back until the model choices are complete.

This keeps model selection separate from the final evaluation.

In [5]:
X_reg = churn[reg_features].copy()
y_reg = churn[reg_target].astype(float).copy()

X_dev_reg, X_test_reg, y_dev_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

X_train_reg, X_val_reg, y_train_reg, y_val_reg = train_test_split(
    X_dev_reg,
    y_dev_reg,
    test_size=0.25,
    random_state=RANDOM_STATE,
)

reg_split_summary = pd.DataFrame({
    "Rows": [len(X_train_reg), len(X_val_reg), len(X_test_reg)],
    "Mean OrderCount": [
        y_train_reg.mean(),
        y_val_reg.mean(),
        y_test_reg.mean(),
    ],
}, index=["Training", "Validation", "Test"])

display(reg_split_summary)

,Rows,Mean OrderCount
Training,3378,2.9313
Validation,1126,3.1252
Test,1126,2.8899


In [6]:
def make_preprocessor(numeric_columns):
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            sparse_output=False,
        )),
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipe, numeric_columns),
        ("categorical", categorical_pipe, categorical_features),
    ])


def regression_metrics(y_true, prediction):
    return {
        "RMSE": mean_squared_error(y_true, prediction) ** 0.5,
        "MAE": mean_absolute_error(y_true, prediction),
        "R-squared": r2_score(y_true, prediction),
    }


reg_preprocessor = make_preprocessor(reg_numeric_features)
X_train_reg_t = reg_preprocessor.fit_transform(X_train_reg)
X_val_reg_t = reg_preprocessor.transform(X_val_reg)

print(f"Transformed regression feature count: {X_train_reg_t.shape[1]}")

Transformed regression feature count: 25


## 5. How the first boosting step works

For squared-error regression, gradient boosting begins with a simple starting prediction that is closely related to the mean response. The remaining error becomes the residual that the next weak learner tries to explain.

I reproduce the first step manually:

1. predict the training mean;
2. calculate the residuals;
3. fit a depth-1 decision stump to those residuals;
4. multiply the stump prediction by the learning rate;
5. add that correction to the starting prediction.

The fitted gradient-boosting model repeats this process across many trees.

In [7]:
initial_prediction = float(y_train_reg.mean())
initial_vector = np.repeat(initial_prediction, len(y_train_reg))
first_residual = y_train_reg.to_numpy() - initial_vector

stump = DecisionTreeRegressor(
    max_depth=1,
    min_samples_leaf=20,
    random_state=RANDOM_STATE,
)
stump.fit(X_train_reg_t, first_residual)
stump_correction = stump.predict(X_train_reg_t)

first_step_rows = []
for learning_rate in [1.0, 0.5, 0.1]:
    updated_prediction = initial_vector + learning_rate * stump_correction
    first_step_rows.append({
        "Learning rate": learning_rate,
        "Training RMSE after first stump":
            mean_squared_error(y_train_reg, updated_prediction) ** 0.5,
    })

first_step_results = pd.DataFrame(first_step_rows)

print(f"Initial mean prediction: {initial_prediction:.4f}")
print(
    "Initial training RMSE before adding a weak learner: "
    f"{mean_squared_error(y_train_reg, initial_vector) ** 0.5:.4f}"
)

display(first_step_results)

diagnostic_table = pd.DataFrame({
    "Actual OrderCount": y_train_reg.iloc[:8].to_numpy(),
    "Initial prediction": initial_vector[:8],
    "Residual before stump": first_residual[:8],
    "Stump residual prediction": stump_correction[:8],
})
display(diagnostic_table)

Initial mean prediction: 2.9313
Initial training RMSE before adding a weak learner: 2.8436


,Learning rate,Training RMSE after first stump
0,1.0000,2.2639
1,0.5000,2.4218
2,0.1000,2.7429


,Actual OrderCount,Initial prediction,Residual before stump,Stump residual prediction
0,5.0000,2.9313,2.0687,-0.7793
1,5.0000,2.9313,2.0687,-0.7793
2,2.0000,2.9313,-0.9313,-0.7793
3,2.0000,2.9313,-0.9313,-0.7793
4,4.0000,2.9313,1.0687,-0.7793
5,2.0000,2.9313,-0.9313,-0.7793
6,3.0000,2.9313,0.0687,-0.7793
7,2.0000,2.9313,-0.9313,-0.7793


In [8]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(
    first_step_results["Learning rate"].astype(str),
    first_step_results["Training RMSE after first stump"],
)
ax.set_xlabel("Learning rate")
ax.set_ylabel("Training RMSE after first weak learner")
ax.set_title("A smaller learning rate makes a smaller first correction")
ax.grid(axis="y")
plt.tight_layout()
plt.show()

<Figure size 700x500 with 1 Axes>

### First-step result

With only one stump, a learning rate of 1 makes the largest correction and therefore produces the largest immediate reduction in training error.

A smaller learning rate changes the prediction more gradually, so additional trees are needed before the model can reach the same level of fit. The learning rate therefore only makes sense when considered together with the number of estimators.

## 6. Learning rate and number of estimators

The learning rate controls how much each new tree changes the running prediction. Smaller values make more conservative updates and usually require more trees, while larger values learn faster but can become too aggressive when many trees are added.

I hold weak-learner depth at 2 and compare several learning-rate and estimator combinations using validation RMSE.

In [9]:
learning_rates = [0.05, 0.10, 0.20]
estimator_counts = [50, 100, 200, 400]

lr_rows = []

# Fit one maximum-length sequence per learning rate, then inspect staged
# predictions instead of refitting a separate model for every estimator count.
for learning_rate in learning_rates:
    model = GradientBoostingRegressor(
        random_state=RANDOM_STATE,
        learning_rate=learning_rate,
        n_estimators=max(estimator_counts),
        max_depth=2,
        min_samples_leaf=10,
        subsample=1.0,
    )
    model.fit(X_train_reg_t, y_train_reg)

    staged = list(model.staged_predict(X_val_reg_t))
    for n_estimators in estimator_counts:
        prediction = staged[n_estimators - 1]
        lr_rows.append({
            "Learning rate": learning_rate,
            "Estimators": n_estimators,
            "Validation RMSE":
                mean_squared_error(y_val_reg, prediction) ** 0.5,
            "Validation R-squared":
                r2_score(y_val_reg, prediction),
        })

lr_results = pd.DataFrame(lr_rows)
display(
    lr_results.sort_values("Validation RMSE").head(12)
)

,Learning rate,Estimators,Validation RMSE,Validation R-squared
11,0.2000,400,1.8562,0.6172
7,0.1000,400,1.8924,0.6021
10,0.2000,200,1.8973,0.6000
9,0.2000,100,1.9411,0.5813
6,0.1000,200,1.9421,0.5809
3,0.0500,400,1.9436,0.5802
8,0.2000,50,1.9890,0.5604
5,0.1000,100,1.9922,0.5590
2,0.0500,200,1.9951,0.5577
4,0.1000,50,2.0478,0.5340


In [10]:
fig, ax = plt.subplots(figsize=(9, 5))

for learning_rate in learning_rates:
    subset = lr_results[
        lr_results["Learning rate"] == learning_rate
    ].sort_values("Estimators")

    ax.plot(
        subset["Estimators"],
        subset["Validation RMSE"],
        marker="o",
        label=f"learning_rate={learning_rate}",
    )

ax.set_xlabel("Number of estimators")
ax.set_ylabel("Validation RMSE")
ax.set_title("Learning rate and estimator count trade off against each other")
ax.legend()
ax.grid()
plt.tight_layout()
plt.show()

<Figure size 900x500 with 1 Axes>

### A second view of the learning-rate tradeoff

The line chart shows how each learning rate evolves as trees are added. The heatmap below makes the full comparison easier to scan: lower RMSE indicates the stronger validation combinations.

In [ ]:
# Heatmap of validation RMSE across learning rate and estimator count
lr_pivot = lr_results.pivot(
    index="Learning rate",
    columns="Estimators",
    values="Validation RMSE",
)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(lr_pivot.values, aspect="auto")

ax.set_xticks(np.arange(len(lr_pivot.columns)))
ax.set_xticklabels([str(int(x)) for x in lr_pivot.columns])
ax.set_yticks(np.arange(len(lr_pivot.index)))
ax.set_yticklabels([f"{x:.2f}" for x in lr_pivot.index])

ax.set_xlabel("Number of estimators")
ax.set_ylabel("Learning rate")
ax.set_title("Validation RMSE across learning rate and estimator count")

for i in range(len(lr_pivot.index)):
    for j in range(len(lr_pivot.columns)):
        ax.text(
            j,
            i,
            f"{lr_pivot.iloc[i, j]:.2f}",
            ha="center",
            va="center",
        )

fig.colorbar(im, ax=ax, label="Validation RMSE")
plt.tight_layout()
plt.show()

<Figure>

### Learning-rate result

The lower learning rates improve more gradually as estimators are added. Their best validation results occur only after hundreds of trees.

The curves show that learning rate and estimator count are not separate decisions: shrinking each update changes how many boosting stages are needed.

## 7. Tree depth and the bias-variance tradeoff

Each boosting stage can use a very shallow tree or a more complex one. Shallow trees may miss interactions, while deeper trees can fit increasingly specific residual patterns.

To see where that flexibility stops helping, I hold the learning rate and estimator count fixed and compare training and validation RMSE as depth increases.

In [11]:
depth_rows = []

for depth in [1, 2, 3, 4, 6]:
    model = GradientBoostingRegressor(
        random_state=RANDOM_STATE,
        learning_rate=0.10,
        n_estimators=400,
        max_depth=depth,
        min_samples_leaf=5,
        subsample=1.0,
    )
    model.fit(X_train_reg_t, y_train_reg)

    train_prediction = model.predict(X_train_reg_t)
    val_prediction = model.predict(X_val_reg_t)

    depth_rows.append({
        "Depth": depth,
        "Training RMSE":
            mean_squared_error(y_train_reg, train_prediction) ** 0.5,
        "Validation RMSE":
            mean_squared_error(y_val_reg, val_prediction) ** 0.5,
        "Validation R-squared":
            r2_score(y_val_reg, val_prediction),
    })

depth_results = pd.DataFrame(depth_rows)
display(depth_results)

,Depth,Training RMSE,Validation RMSE,Validation R-squared
0,1,1.8375,2.0682,0.5247
1,2,1.4682,1.8963,0.6004
2,3,1.1490,1.8242,0.6302
3,4,0.8670,1.7474,0.6607
4,6,0.3407,1.7681,0.6526


In [12]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    depth_results["Depth"],
    depth_results["Training RMSE"],
    marker="o",
    label="Training RMSE",
)
ax.plot(
    depth_results["Depth"],
    depth_results["Validation RMSE"],
    marker="o",
    label="Validation RMSE",
)

ax.set_xlabel("Weak-learner tree depth")
ax.set_ylabel("RMSE")
ax.set_title("Deeper weak learners reduce training error faster than validation error")
ax.legend()
ax.grid()
plt.tight_layout()
plt.show()

<Figure size 800x500 with 1 Axes>

### How quickly the train-validation gap grows

The RMSE gap below isolates the generalization issue. A larger bar means the model is fitting the training data much better than the validation data.

In [ ]:
# Generalization gap as weak-learner depth increases
depth_gap = depth_results.copy()
depth_gap["RMSE gap"] = (
    depth_gap["Validation RMSE"]
    - depth_gap["Training RMSE"]
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(
    depth_gap["Depth"].astype(str),
    depth_gap["RMSE gap"],
)
ax.set_xlabel("Weak-learner depth")
ax.set_ylabel("Validation RMSE - training RMSE")
ax.set_title("Generalization gap widens as weak learners become deeper")
ax.grid(axis="y")
plt.tight_layout()
plt.show()

<Figure>

### Depth result

Training RMSE continues to fall as the weak learners become deeper. Validation RMSE improves through moderate depth, but the gains become much smaller while training error keeps dropping.

By depth 6 and beyond, the widening train-validation gap suggests that the extra tree complexity is fitting training-specific structure more than it is improving predictions for unseen customers. Moderate-depth learners are therefore more defensible for this dataset.

## 8. Regularization with subsampling and leaf size

I next test two ways to constrain the boosted model:

- **subsample:** each boosting stage is fitted on only a fraction of the training rows;
- **min_samples_leaf:** each terminal leaf must contain enough observations to avoid very specific residual rules.

The comparison keeps the rest of the boosting setup fixed so the effect of these controls is easier to see.

In [13]:
regularization_rows = []

# First vary stochastic subsampling while holding leaf size fixed.
for subsample in [1.0, 0.8, 0.6]:
    model = GradientBoostingRegressor(
        random_state=RANDOM_STATE,
        learning_rate=0.10,
        n_estimators=400,
        max_depth=4,
        min_samples_leaf=5,
        subsample=subsample,
    )
    model.fit(X_train_reg_t, y_train_reg)
    train_prediction = model.predict(X_train_reg_t)
    val_prediction = model.predict(X_val_reg_t)
    regularization_rows.append({
        "Subsample": subsample,
        "Minimum leaf size": 5,
        "Training RMSE": mean_squared_error(y_train_reg, train_prediction) ** 0.5,
        "Validation RMSE": mean_squared_error(y_val_reg, val_prediction) ** 0.5,
        "Validation R-squared": r2_score(y_val_reg, val_prediction),
    })

# Then vary leaf size at the strongest subsample level.
for min_samples_leaf in [1, 5, 15, 30]:
    model = GradientBoostingRegressor(
        random_state=RANDOM_STATE,
        learning_rate=0.10,
        n_estimators=400,
        max_depth=4,
        min_samples_leaf=min_samples_leaf,
        subsample=0.60,
    )
    model.fit(X_train_reg_t, y_train_reg)
    train_prediction = model.predict(X_train_reg_t)
    val_prediction = model.predict(X_val_reg_t)
    regularization_rows.append({
        "Subsample": 0.60,
        "Minimum leaf size": min_samples_leaf,
        "Training RMSE": mean_squared_error(y_train_reg, train_prediction) ** 0.5,
        "Validation RMSE": mean_squared_error(y_val_reg, val_prediction) ** 0.5,
        "Validation R-squared": r2_score(y_val_reg, val_prediction),
    })

regularization_results = (
    pd.DataFrame(regularization_rows)
    .drop_duplicates(subset=["Subsample", "Minimum leaf size"])
)
display(regularization_results.sort_values("Validation RMSE"))

,Subsample,Minimum leaf size,Training RMSE,Validation RMSE,Validation R-squared
2,0.6000,5,0.8101,1.7051,0.6769
1,0.8000,5,0.8081,1.7356,0.6653
0,1.0000,5,0.8670,1.7474,0.6607
5,0.6000,15,0.9962,1.7533,0.6584
3,0.6000,1,0.7039,1.7570,0.6570
6,0.6000,30,1.1350,1.7711,0.6514


In [14]:
fig, ax = plt.subplots(figsize=(8, 5))

leaf_subset = regularization_results[
    regularization_results["Subsample"] == 0.60
].sort_values("Minimum leaf size")

ax.plot(
    leaf_subset["Minimum leaf size"],
    leaf_subset["Validation RMSE"],
    marker="o",
    label="leaf-size effect at subsample=0.60",
)

subsample_subset = regularization_results[
    regularization_results["Minimum leaf size"] == 5
].sort_values("Subsample")

ax.plot(
    subsample_subset["Subsample"] * 30,
    subsample_subset["Validation RMSE"],
    marker="o",
    label="subsample effect (x-axis scaled for display)",
)

ax.set_xlabel("Regularization setting index / minimum leaf size")
ax.set_ylabel("Validation RMSE")
ax.set_title("Regularization changes validation error")
ax.legend()
ax.grid()
plt.tight_layout()
plt.show()

<Figure size 800x500 with 1 Axes>

### Training fit versus validation fit

This view makes the regularization tradeoff more direct. Settings farther to the left fit the training data more closely; settings lower on the chart perform better on validation data.

In [ ]:
# Training versus validation performance for the regularization settings
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(
    regularization_results["Training RMSE"],
    regularization_results["Validation RMSE"],
    s=70,
)

for _, row in regularization_results.iterrows():
    ax.annotate(
        f"s={row['Subsample']:.1f}, leaf={int(row['Minimum leaf size'])}",
        (row["Training RMSE"], row["Validation RMSE"]),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=8,
    )

ax.set_xlabel("Training RMSE")
ax.set_ylabel("Validation RMSE")
ax.set_title("Regularization trades training fit for validation performance")
ax.grid()
plt.tight_layout()
plt.show()

<Figure>

### Regularization result

The best validation RMSE occurs with **subsample = 0.6** and **minimum leaf size = 5**.

Using only part of the training data at each boosting stage improved validation performance relative to the fully deterministic version. In this case, the added randomness helped the model generalize better.

## 9. Final model selection

The learning-rate experiment initially used depth-2 weak learners. After the depth and regularization comparisons identified a stronger structure, I make one final validation-only check to see whether extending the sequence from 400 to 600 estimators still lowers RMSE.

The test set remains untouched during this decision.

In [15]:
final_stage_rows = []

for n_estimators in [400, 600]:
    model = GradientBoostingRegressor(
        random_state=RANDOM_STATE,
        learning_rate=0.10,
        n_estimators=n_estimators,
        max_depth=4,
        min_samples_leaf=5,
        subsample=0.60,
    )
    model.fit(X_train_reg_t, y_train_reg)
    prediction = model.predict(X_val_reg_t)
    final_stage_rows.append({
        "Estimators": n_estimators,
        "Validation RMSE": mean_squared_error(y_val_reg, prediction) ** 0.5,
        "Validation R-squared": r2_score(y_val_reg, prediction),
    })

final_stage_check = pd.DataFrame(final_stage_rows)
display(final_stage_check)

,Estimators,Validation RMSE,Validation R-squared
0,400,1.7051,0.6769
1,600,1.6971,0.6800


The 600-estimator model has the lower validation RMSE with the selected depth and regularization settings, so I use that configuration for the final refit.

In [16]:
final_reg_preprocessor = make_preprocessor(reg_numeric_features)
X_dev_reg_t = final_reg_preprocessor.fit_transform(X_dev_reg)
X_test_reg_t = final_reg_preprocessor.transform(X_test_reg)

final_reg_model = GradientBoostingRegressor(
    random_state=RANDOM_STATE,
    learning_rate=0.10,
    n_estimators=600,
    max_depth=4,
    min_samples_leaf=5,
    subsample=0.60,
)
final_reg_model.fit(X_dev_reg_t, y_dev_reg)

reg_train_prediction = final_reg_model.predict(X_dev_reg_t)
reg_test_prediction = final_reg_model.predict(X_test_reg_t)

regression_summary = pd.DataFrame([
    {
        "Split": "Development",
        **regression_metrics(y_dev_reg, reg_train_prediction),
    },
    {
        "Split": "Test",
        **regression_metrics(y_test_reg, reg_test_prediction),
    },
])

display(regression_summary)

,Split,RMSE,MAE,R-squared
0,Development,0.7721,0.4895,0.9284
1,Test,1.4011,0.8251,0.7592


In [17]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test_reg, reg_test_prediction)

limits = [
    min(y_test_reg.min(), reg_test_prediction.min()),
    max(y_test_reg.max(), reg_test_prediction.max()),
]
ax.plot(limits, limits)

ax.set_xlim(limits)
ax.set_ylim(limits)
ax.set_xlabel("Actual OrderCount")
ax.set_ylabel("Predicted OrderCount")
ax.set_title("Gradient-boosting regression: actual versus predicted")
ax.grid()
plt.tight_layout()
plt.show()

<Figure size 600x600 with 1 Axes>

### Improvement over a simple baseline

A model should beat a simple reference point before its complexity is worth discussing. Here I compare the final test RMSE with a mean-only predictor that gives every test customer the average development-set `OrderCount`.

In [ ]:
# Mean-only baseline versus the final gradient-boosting model
baseline_prediction = np.repeat(
    y_dev_reg.mean(),
    len(y_test_reg),
)
baseline_rmse = (
    mean_squared_error(
        y_test_reg,
        baseline_prediction,
    ) ** 0.5
)
final_rmse = (
    mean_squared_error(
        y_test_reg,
        reg_test_prediction,
    ) ** 0.5
)

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(
    ["Mean-only baseline", "Gradient boost"],
    [baseline_rmse, final_rmse],
)
ax.set_ylabel("Test RMSE")
ax.set_title("Gradient boosting improves substantially over a mean-only baseline")
ax.grid(axis="y")

for i, value in enumerate([baseline_rmse, final_rmse]):
    ax.text(
        i,
        value,
        f"{value:.2f}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

<Figure>

### Residual check

The actual-versus-predicted chart shows overall agreement. The residual plot below is more useful for diagnosing whether the model systematically misses particular parts of the prediction range.

In [ ]:
# Residual pattern on the untouched test set
test_residuals = (
    y_test_reg.to_numpy()
    - reg_test_prediction
)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    reg_test_prediction,
    test_residuals,
    alpha=0.55,
)
ax.axhline(0, linestyle="--")
ax.set_xlabel("Predicted OrderCount")
ax.set_ylabel("Residual (actual - predicted)")
ax.set_title("Residuals across the range of predicted OrderCount")
ax.grid()
plt.tight_layout()
plt.show()

<Figure>

### Where are the largest errors?

Because `OrderCount` is discrete and right-skewed, average performance can hide where the model struggles. The next chart shows mean absolute error by actual order count for levels with at least eight test observations.

In [ ]:
# Mean absolute error by actual OrderCount
error_by_count = pd.DataFrame({
    "Actual OrderCount": y_test_reg.to_numpy(),
    "Absolute error": np.abs(
        y_test_reg.to_numpy()
        - reg_test_prediction
    ),
})

error_by_count = (
    error_by_count
    .groupby("Actual OrderCount", as_index=False)
    ["Absolute error"]
    .agg(["mean", "count"])
    .reset_index()
)

error_by_count = error_by_count[
    error_by_count["count"] >= 8
]

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    error_by_count["Actual OrderCount"]
    .astype(int)
    .astype(str),
    error_by_count["mean"],
)
ax.set_xlabel("Actual OrderCount")
ax.set_ylabel("Mean absolute error")
ax.set_title("Prediction error is larger for some order-count levels")
ax.grid(axis="y")
plt.tight_layout()
plt.show()

<Figure>

### Final regression result

The selected gradient-boosting regressor reaches test RMSE of about **1.40**, MAE of about **0.83**, and test \(R^2\) of about **0.76**.

The development fit is stronger than the test fit, so some variance remains even after regularization. I therefore treat the test result as useful predictive evidence, not as a reason to keep increasing depth or the number of estimators.

## 10. Feature importance

Gradient boosting does not produce one coefficient per predictor. Instead, feature importance summarizes how often and how effectively variables were used to reduce error across the trees.

These values describe predictive contribution inside this fitted model; they should not be interpreted as causal effects.

In [18]:
reg_feature_names = (
    final_reg_preprocessor.get_feature_names_out()
)

reg_importance = pd.DataFrame({
    "Feature": reg_feature_names,
    "Importance": final_reg_model.feature_importances_,
}).sort_values("Importance", ascending=False)

reg_importance["Feature"] = (
    reg_importance["Feature"]
    .str.replace("numeric__", "", regex=False)
    .str.replace("categorical__", "", regex=False)
    .str.replace("PreferredLoginDevice_", "LoginDevice = ", regex=False)
    .str.replace("PreferredPaymentMode_", "PaymentMode = ", regex=False)
    .str.replace("PreferedOrderCat_", "OrderCat = ", regex=False)
)

top_reg_importance = (
    reg_importance.head(15)
    .sort_values("Importance")
)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(
    top_reg_importance["Feature"],
    top_reg_importance["Importance"],
)
ax.set_xlabel("Feature importance")
ax.set_ylabel("Feature")
ax.set_title("Most important predictors for OrderCount")
ax.grid(axis="x")
plt.tight_layout()
plt.show()

display(reg_importance.head(15))

<Figure size 900x600 with 1 Axes>

,Feature,Importance
9,CouponUsed,0.3787
11,CashbackAmount,0.2455
10,DaySinceLastOrder,0.1066
2,WarehouseToHome,0.0484
0,Tenure,0.0483
6,NumberOfAddress,0.0326
8,OrderAmountHikeFromlastYear,0.0290
5,SatisfactionScore,0.0201
4,NumberOfDeviceRegistered,0.0170
1,CityTier,0.0115


`CouponUsed`, `CashbackAmount`, and `DaySinceLastOrder` contribute most strongly to the final model, followed by customer-history and delivery-related variables.

The pattern suggests that ordering activity is closely associated with recent engagement and transaction incentives. The ranking is specific to this fitted model and does not show that changing any one variable would cause order volume to change.

## 11. Main findings

### Residual correction is visible in the first boosting step

The manual first step shows how the model starts from a simple prediction, fits a stump to the remaining error, and uses the learning rate to control how much of that correction is added.

### Learning rate and estimator count work together

Smaller learning rates need more trees before they become competitive on validation RMSE. Larger rates learn faster because each weak learner has more influence. Neither parameter should be interpreted in isolation.

### Depth made the overfitting pattern visible

Deeper trees reduced training RMSE rapidly, but validation performance did not improve at the same rate. The widening gap showed that added weak-learner complexity eventually began fitting training-specific structure.

### Regularization improved validation performance

A 60% subsample with a minimum leaf size of 5 produced the strongest validation result among the tested regularization settings. The stochastic version therefore generalized better than the fully deterministic alternative in this experiment.

### Final performance

The selected model reached test RMSE of about **1.40**, MAE of about **0.83**, and test \(R^2\) of about **0.76**.

Because the development fit remains stronger than the test fit, I would validate the model on additional splits or a later-period sample before treating this result as stable enough for operational use.

## 12. Limitations

- Too many estimators or overly deep weak learners can still overfit.
- The final settings came from targeted validation experiments rather than an exhaustive hyperparameter search.
- Validation and test results come from one development split; repeated cross-validation would provide a more stable estimate.
- Impurity-based feature importance can favor variables with more possible split points.
- `OrderCount` is discrete even though the regression model returns continuous predictions.
- Customer rows are treated as independent observations.
- The results describe predictive associations rather than causal effects.

## 13. Reproducibility notes

- Random seed: 42
- Data file: `data/churn_clean.csv`
- Regression target: `OrderCount`
- Train / validation / test split: 60% / 20% / 20%
- Numeric preprocessing: median imputation
- Categorical preprocessing: most-frequent imputation + one-hot encoding
- Feature scaling: not used because gradient boosting is tree based
- Learning rates tested: 0.05, 0.10, 0.20
- Estimator counts examined: 50, 100, 200, 400; final regularized check at 600
- Weak-learner depths examined: 1, 2, 3, 4, 6
- Subsamples examined: 1.0, 0.8, 0.6
- Minimum leaf sizes examined: 1, 5, 15, 30
- Model-selection metric: validation RMSE
- Test set evaluated only after model choices were fixed